# Train Tiny YOLO From Scratch on Kaggle GPU

Notebook này clone repo XLA từ GitHub, cài dependency, dùng GPU Kaggle để train và lưu `models/best.pth`.

In [ ]:
# Dán link GitHub repo XLA của bạn vào đây.
# Ví dụ: REPO_URL = "https://github.com/username/XLA.git"
REPO_URL = "https://github.com/huyvanzzz/XLA.git"
BRANCH = "main"

# Nếu bạn upload dataset public thành Kaggle Dataset, sửa đường dẫn này.
# Cấu trúc cần có: public/classes.json, public/train/images, public/val/images, public/annotations/*.json
KAGGLE_PUBLIC_DIR = "/kaggle/input/xla-object-detection/public"

WORK_DIR = "/kaggle/working/XLA"

In [ ]:
import os, shutil, subprocess, textwrap
from pathlib import Path

assert REPO_URL != "PASTE_YOUR_XLA_GITHUB_URL_HERE", "Hãy dán link GitHub repo XLA vào REPO_URL trước."

if Path(WORK_DIR).exists():
    shutil.rmtree(WORK_DIR)

subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, WORK_DIR], check=True)
os.chdir(WORK_DIR)
print("Repo cloned to", WORK_DIR)

In [ ]:
# Kaggle sometimes ships a very new PyTorch build that does not support Tesla P100 (sm_60).
# This build is compatible with P100 and still works for T4/P100 Kaggle GPUs.
!python -m pip uninstall -y -q torch torchvision torchaudio
!python -m pip install -q --no-cache-dir --index-url https://download.pytorch.org/whl/cu121 torch==2.4.1+cu121
!python -m pip install -q -r requirements.txt

In [ ]:
from pathlib import Path
import shutil

src_public = Path(KAGGLE_PUBLIC_DIR)
dst_public = Path(WORK_DIR) / "public"

if src_public.exists():
    if dst_public.exists():
        shutil.rmtree(dst_public)
    shutil.copytree(src_public, dst_public)
    print("Copied dataset from", src_public)
elif dst_public.exists():
    print("Using public/ already inside repo")
else:
    raise FileNotFoundError(f"Không tìm thấy dataset ở {src_public}. Hãy upload public/ lên Kaggle Dataset rồi sửa KAGGLE_PUBLIC_DIR.")

In [ ]:
import torch
print("torch", torch.__version__)
print("cuda available", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/ \
  --config ./configs/default.yaml

In [ ]:
!python predict.py \
  --image_dir ./public/val/images \
  --output ./val_predictions.json \
  --checkpoint ./models/best.pth \
  --config ./configs/default.yaml

!python public/tools/evaluate_predictions.py \
  --ground_truth ./public/annotations/val.json \
  --predictions ./val_predictions.json \
  --output ./val_score.json

!cat ./val_score.json

In [ ]:
from pathlib import Path
import shutil

artifact_dir = Path('/kaggle/working/artifacts')
artifact_dir.mkdir(exist_ok=True)
for name in ['best.pth', 'last.pth']:
    src = Path('./models') / name
    if src.exists():
        shutil.copy2(src, artifact_dir / name)
for name in ['val_predictions.json', 'val_score.json']:
    src = Path(name)
    if src.exists():
        shutil.copy2(src, artifact_dir / name)
print('Artifacts:', sorted(p.name for p in artifact_dir.iterdir()))